# Asignación #4 — KNN y Random Forest

**Universidad Tecnológica de Panamá**  
**Asignatura:** Inteligencia Artificial  
**Profesor:** Ing. Manuel Paz  
**Dataset:** Pumpkin Seeds Dataset

## Objetivo

Implementar y comparar dos modelos de clasificación: **K-Nearest Neighbors (KNN)** y **Random Forest**.  
El modelo debe predecir la variedad de una semilla de calabaza a partir de sus características morfológicas.

## Pasos realizados en el notebook

1. Carga del dataset.
2. Análisis exploratorio de datos.
3. Revisión de calidad de datos.
4. Limpieza y preparación de variables.
5. División en datos de entrenamiento y prueba.
6. Escalado de características.
7. Entrenamiento de KNN.
8. Entrenamiento de Random Forest.
9. Evaluación con métricas.
10. Comparación e interpretación de resultados.

> Para ejecutar el notebook, el archivo `Pumpkin_Seeds_Dataset.xlsx` debe estar en la misma carpeta.


## Paso 1 — Importar librerías

Primero se importan las librerías necesarias para trabajar con los datos, hacer gráficas, preparar la información, entrenar los modelos y evaluar los resultados.

En esta parte se usa:

- `pandas` y `numpy` para manejar datos.
- `matplotlib` y `seaborn` para visualizaciones.
- `scikit-learn` para dividir datos, escalar variables, crear modelos y calcular métricas.
- `RANDOM_STATE = 42` para que los resultados sean reproducibles.


In [ ]:
%pip install pandas numpy matplotlib scikit-learn openpyxl seaborn

In [ ]:
# Librerías para manejo de datos
import numpy as np
import pandas as pd

# Librerías para gráficos
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de preparación
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import StandardScaler, LabelEncoder

# Modelos
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier

# Métricas
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
                             classification_report, confusion_matrix, ConfusionMatrixDisplay)

import warnings
warnings.filterwarnings("ignore")

# Configuración general
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams["figure.dpi"] = 110
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

print("Librerías importadas correctamente.")


## Paso 2 — Cargar los datos

En este paso se carga el archivo `Pumpkin_Seeds_Dataset.xlsx` con `pandas`.

Después de cargarlo, se revisa:

- Cantidad de filas.
- Cantidad de columnas.
- Primeras observaciones del dataset.

Esto permite comprobar que el archivo se leyó correctamente antes de hacer cualquier análisis.


In [ ]:
# El archivo .xlsx debe estar en la misma carpeta que este notebook
df = pd.read_excel("Pumpkin_Seeds_Dataset.xlsx")

print(f"El conjunto de datos tiene {df.shape[0]} filas (semillas) y {df.shape[1]} columnas.")
df.head()


## Paso 3 — Análisis Exploratorio de Datos (EDA)

El análisis exploratorio permite conocer cómo vienen los datos antes de entrenar los modelos.

En esta primera revisión se observa:

- Tipos de datos.
- Columnas disponibles.
- Cantidad de valores no nulos.
- Variable objetivo (`Class`).

Este paso es importante porque ayuda a detectar problemas desde el inicio.


In [ ]:
# Tipo de cada columna, conteo de no-nulos y memoria
df.info()


### Paso 3.1 — Estadísticos descriptivos

Aquí se calculan estadísticas básicas de las variables numéricas: media, desviación estándar, mínimo, máximo y cuartiles.

Esto ayuda a conocer el rango de cada variable y a detectar si tienen escalas muy diferentes.  
En este dataset, variables como `Area` y `Convex_Area` tienen valores muy grandes, mientras que otras como `Eccentricity` o `Compactness` están entre 0 y 1. Por eso más adelante se aplica escalado.


In [ ]:
# Resumen estadístico de las variables numéricas
df.describe().T


### Paso 3.2 — Revisión de nulos y duplicados

En esta parte se revisa si existen valores faltantes o filas duplicadas.

Los valores nulos pueden afectar el entrenamiento del modelo si no se corrigen.  
Los duplicados también pueden generar sesgo, porque el modelo podría aprender registros repetidos.


In [ ]:
print("Valores nulos por columna:")
print(df.isnull().sum())
print(f"\nFilas duplicadas: {df.duplicated().sum()}")


### Paso 3.3 — Balance de clases

Ahora se revisa la distribución de la columna `Class`.

Esto es necesario porque, si una clase tuviera muchas más observaciones que la otra, el accuracy podría ser engañoso.  
En este caso, las clases están bastante balanceadas, aproximadamente 52% y 48%, por lo que el problema no presenta un desbalance fuerte.


In [ ]:
conteo = df["Class"].value_counts()
print(conteo)
print("\nProporción:")
print((df["Class"].value_counts(normalize=True)*100).round(2))

plt.figure(figsize=(6,4))
ax = sns.countplot(x="Class", data=df)
ax.set_title("Distribución de la variable objetivo (Class)")
ax.set_ylabel("Cantidad de semillas")
for p in ax.patches:
    ax.annotate(int(p.get_height()), (p.get_x()+p.get_width()/2, p.get_height()),
                ha="center", va="bottom")
plt.tight_layout(); plt.show()


### Paso 3.4 — Distribución de características

Se grafican histogramas para observar cómo se distribuye cada variable numérica.

Con esto se puede ver si las variables están concentradas en ciertos rangos, si tienen asimetrías o si existen posibles valores extremos.


In [ ]:
features = [c for c in df.columns if c != "Class"]
df[features].hist(figsize=(15,10), bins=30, edgecolor="black")
plt.suptitle("Distribución de cada característica", y=1.02, fontsize=14)
plt.tight_layout(); plt.show()


### Paso 3.5 — Comparación de variables por clase

En esta sección se usan boxplots para comparar cada característica según la variedad de semilla.

Este análisis ayuda a identificar qué variables parecen separar mejor las clases.  
Si las cajas de una variable se ven muy diferentes entre clases, esa variable puede ser útil para la clasificación.


In [ ]:
fig, axes = plt.subplots(3, 4, figsize=(16,11))
for ax, col in zip(axes.ravel(), features):
    sns.boxplot(x="Class", y=col, data=df, ax=ax)
    ax.set_title(col); ax.set_xlabel("")
plt.suptitle("Boxplots de cada característica según la variedad", y=1.01, fontsize=14)
plt.tight_layout(); plt.show()


### Paso 3.6 — Correlación entre variables

Se calcula una matriz de correlación para ver qué variables numéricas están relacionadas entre sí.

Esto permite identificar variables que se comportan de forma parecida.  
También ayuda a entender mejor la estructura del dataset antes de entrenar los modelos.


In [ ]:
plt.figure(figsize=(11,9))
corr = df[features].corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0,
            square=True, cbar_kws={"shrink":.8})
plt.title("Matriz de correlación entre características")
plt.tight_layout(); plt.show()


**¿Qué observamos?**  
Hay grupos de variables con correlación muy alta (cercana a 1), como `Area`, `Convex_Area` y `Equiv_Diameter`. Esto es **multicolinealidad**: variables que aportan información redundante. KNN se ve perjudicado por esto (cuenta la misma información varias veces en la distancia), mientras que Random Forest es robusto frente a ella. Mantenemos todas las variables para comparar los modelos en igualdad de condiciones.

## Paso 4 — Limpieza y preparación de datos

Después del EDA, se preparan los datos para poder usarlos en los modelos.

En esta parte se corrigen valores nulos usando la mediana de cada columna numérica.  
Se usa la mediana porque es más resistente a valores extremos que el promedio.


In [ ]:
nulos_antes = df.isnull().sum().sum()
print(f"Valores nulos antes de imputar: {nulos_antes}")

# Imputación por mediana solo en columnas numéricas
for col in features:
    if df[col].isnull().any():
        mediana = df[col].median()
        df[col] = df[col].fillna(mediana)
        print(f"  - '{col}' imputada con su mediana = {mediana:.4f}")

print(f"Valores nulos después de imputar: {df.isnull().sum().sum()}")


### Paso 4.1 — Codificación de la variable objetivo

La columna `Class` contiene texto, pero los modelos de scikit-learn trabajan con valores numéricos.

Por eso se transforma la variable objetivo en números:

- `0` para una clase.
- `1` para la otra clase.

También se guarda el mapa de clases para poder interpretar los resultados al final.


In [ ]:
le = LabelEncoder()
df["Class_encoded"] = le.fit_transform(df["Class"])

# Diccionario de mapeo para interpretar resultados más adelante
mapa_clases = dict(zip(le.transform(le.classes_), le.classes_))
print("Codificación de clases:", mapa_clases)
df[["Class", "Class_encoded"]].drop_duplicates().reset_index(drop=True)


### Paso 4.2 — Separar variables predictoras y variable objetivo

En este paso se separan los datos en:

- `X`: variables predictoras, es decir, las características morfológicas de las semillas.
- `y`: variable objetivo, que es la clase codificada.

Esta separación es necesaria porque el modelo aprende usando `X` para predecir `y`.


In [ ]:
X = df[features]            # 12 características morfológicas
y = df["Class_encoded"]    # variedad codificada (0 / 1)
print("Forma de X:", X.shape)
print("Forma de y:", y.shape)


### Paso 4.3 — División en entrenamiento y prueba

El dataset se divide en dos partes:

- 80% para entrenamiento.
- 20% para prueba.

También se usa `stratify=y` para mantener la misma proporción de clases en ambos conjuntos.  
Esto permite evaluar el modelo con datos que no fueron usados durante el entrenamiento.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE, stratify=y)

print(f"Entrenamiento: {X_train.shape[0]} muestras")
print(f"Prueba:        {X_test.shape[0]} muestras")
print("\nProporción de clases en train:")
print(y_train.value_counts(normalize=True).round(3).to_dict())
print("Proporción de clases en test: ")
print(y_test.value_counts(normalize=True).round(3).to_dict())


### Paso 4.4 — Escalado de características

Se aplica `StandardScaler` para que las variables tengan media cercana a 0 y desviación estándar cercana a 1.

Este paso es especialmente importante para KNN, porque KNN calcula distancias.  
Si no se escala, las variables con valores grandes, como `Area`, dominarían sobre las demás.

El escalador se ajusta solo con los datos de entrenamiento y luego se aplica al conjunto de prueba para evitar fuga de información.


In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)   # fit SOLO con train
X_test_scaled  = scaler.transform(X_test)        # test solo se transforma

print("Media de las variables en train tras escalar (≈0):")
print(np.round(X_train_scaled.mean(axis=0), 3))
print("\nDesviación estándar tras escalar (≈1):")
print(np.round(X_train_scaled.std(axis=0), 3))


## Paso 5 — Modelo KNN

KNN clasifica una nueva observación mirando las clases de sus vecinos más cercanos.

Antes de entrenar el modelo final, se prueba cuál valor de `k` funciona mejor.  
Para eso se usa validación cruzada con valores impares entre 1 y 29.

El mejor `k` será el que obtenga mayor accuracy promedio durante la validación.


In [ ]:
valores_k = range(1, 31, 2)   # 1, 3, 5, ..., 29
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
scores_cv = []
for k in valores_k:
    modelo_k = KNeighborsClassifier(n_neighbors=k)
    score = cross_val_score(modelo_k, X_train_scaled, y_train, cv=cv, scoring="accuracy").mean()
    scores_cv.append(score)

mejor_k = list(valores_k)[int(np.argmax(scores_cv))]
print(f"Mejor k según validación cruzada: {mejor_k} (accuracy CV = {max(scores_cv):.4f})")

plt.figure(figsize=(8,4))
plt.plot(list(valores_k), scores_cv, marker="o")
plt.axvline(mejor_k, color="red", linestyle="--", label=f"Mejor k = {mejor_k}")
plt.title("Selección de k mediante validación cruzada")
plt.xlabel("Número de vecinos (k)"); plt.ylabel("Accuracy (CV 5-fold)")
plt.legend(); plt.tight_layout(); plt.show()


### Paso 5.1 — Entrenamiento de KNN

Después de escoger el mejor valor de `k`, se entrena el modelo KNN.

Se usa:

- `n_neighbors = mejor_k`
- `weights = "distance"` para dar más peso a los vecinos más cercanos.
- Distancia euclidiana mediante `metric="minkowski"` y `p=2`.

Luego se generan predicciones para entrenamiento y prueba.


In [ ]:
knn = KNeighborsClassifier(n_neighbors=mejor_k, weights="distance", metric="minkowski", p=2)
knn.fit(X_train_scaled, y_train)

# Predicciones
y_pred_knn       = knn.predict(X_test_scaled)
y_train_pred_knn = knn.predict(X_train_scaled)
print(f"Modelo KNN entrenado con k = {mejor_k}.")


## Paso 6 — Modelo Random Forest

Random Forest es un modelo basado en muchos árboles de decisión.

Cada árbol hace una predicción y el bosque combina los resultados.  
Este modelo suele funcionar bien con datos tabulares y permite observar la importancia de las variables.

En este notebook se usa un Random Forest con 200 árboles.


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200, max_depth=None, max_features="sqrt",
    random_state=RANDOM_STATE, n_jobs=-1)
rf.fit(X_train_scaled, y_train)

y_pred_rf       = rf.predict(X_test_scaled)
y_train_pred_rf = rf.predict(X_train_scaled)
print("Modelo Random Forest entrenado con 200 árboles.")


In [ ]:
importancias = pd.Series(rf.feature_importances_, index=features).sort_values()
plt.figure(figsize=(8,6))
importancias.plot(kind="barh")
plt.title("Importancia de las características según Random Forest")
plt.xlabel("Importancia"); plt.tight_layout(); plt.show()
print(importancias.sort_values(ascending=False).round(4))


**¿Qué observamos?**  
Las variables más importantes son las de **forma** (`Aspect_Ration`, `Eccentricity`, `Compactness`, `Roundness`), exactamente las mismas que el EDA identificó como las de mayor separación entre clases. Esto confirma que el modelo aprendió patrones coherentes con la realidad biológica del problema.

---

## Paso 7 — Evaluación de modelos

En esta sección se evalúan KNN y Random Forest usando las métricas solicitadas en la asignación:

- **Accuracy:** proporción total de aciertos.
- **Precision:** qué tan confiables son las predicciones positivas.
- **Recall:** qué tantos casos reales de una clase fueron detectados.
- **F1-score:** equilibrio entre precision y recall.
- **Matriz de confusión:** muestra aciertos y errores por clase.

También se compara el accuracy de entrenamiento contra el de prueba para revisar si hay señales de overfitting.


In [ ]:
nombres_clases = [mapa_clases[0], mapa_clases[1]]

def evaluar(nombre, y_true, y_pred, y_train_true, y_train_pred):
    acc      = accuracy_score(y_true, y_pred)
    acc_tr   = accuracy_score(y_train_true, y_train_pred)
    prec     = precision_score(y_true, y_pred, average="weighted")
    rec      = recall_score(y_true, y_pred, average="weighted")
    f1       = f1_score(y_true, y_pred, average="weighted")
    print("="*60)
    print(f"  {nombre}")
    print("="*60)
    print(f"Accuracy (test) : {acc:.4f}")
    print(f"Accuracy (train): {acc_tr:.4f}   -> brecha train-test: {acc_tr-acc:.4f}")
    print(f"Precision (pond.): {prec:.4f}")
    print(f"Recall    (pond.): {rec:.4f}")
    print(f"F1-score  (pond.): {f1:.4f}\n")
    print("Reporte de clasificación por clase:")
    print(classification_report(y_true, y_pred, target_names=nombres_clases))
    return {"Modelo": nombre, "Accuracy_test": acc, "Accuracy_train": acc_tr,
            "Precision": prec, "Recall": rec, "F1": f1}

res_knn = evaluar("K-Nearest Neighbors (KNN)", y_test, y_pred_knn, y_train, y_train_pred_knn)


Ahora evaluamos el modelo Random Forest con la misma función:

In [ ]:
res_rf = evaluar("Random Forest", y_test, y_pred_rf, y_train, y_train_pred_rf)

### Paso 7.1 — Matrices de confusión

La matriz de confusión permite ver los errores de cada modelo por clase.

La diagonal principal muestra las predicciones correctas.  
Los valores fuera de la diagonal representan errores de clasificación.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13,5))
for ax, (titulo, y_pred) in zip(axes, [("KNN", y_pred_knn), ("Random Forest", y_pred_rf)]):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=nombres_clases)
    disp.plot(ax=ax, cmap="Blues", colorbar=False)
    ax.set_title(f"Matriz de confusión — {titulo}")
    ax.set_xlabel("Predicho"); ax.set_ylabel("Real")
plt.tight_layout(); plt.show()


**¿Qué observamos?**  
En la diagonal están los aciertos; fuera de ella, los errores. Cuanto más concentrada esté la diagonal (y más vacíos los valores fuera de ella), mejor es el modelo. Comparando ambas matrices vemos si algún modelo confunde más una variedad con la otra.

## Paso 8 — Comparación de resultados

En esta parte se comparan las métricas de KNN y Random Forest en una tabla.

También se grafica el desempeño de ambos modelos para ver cuál tuvo mejor resultado en el conjunto de prueba.


In [ ]:
comparacion = pd.DataFrame([res_knn, res_rf]).set_index("Modelo").round(4)
comparacion["Brecha (train-test)"] = (comparacion["Accuracy_train"] - comparacion["Accuracy_test"]).round(4)
display(comparacion)

# Gráfico comparativo de métricas en test
met = ["Accuracy_test", "Precision", "Recall", "F1"]
comparacion[met].plot(kind="bar", figsize=(9,5), rot=0)
plt.title("Comparación de métricas (conjunto de prueba)")
plt.ylabel("Puntuación"); plt.ylim(0,1.05); plt.legend(loc="lower right")
plt.tight_layout(); plt.show()


### Paso 8.1 — Datos para responder las preguntas clave

Aquí se calculan algunos valores necesarios para responder las preguntas de la asignación:

- Mejor modelo por accuracy.
- Brecha entre entrenamiento y prueba.
- Errores por clase en cada modelo.

Estos datos permiten justificar la comparación con números, no solo con opinión.


In [ ]:
# Datos para sustentar las respuestas
mejor_modelo = comparacion["Accuracy_test"].idxmax()
brecha_knn = comparacion.loc["K-Nearest Neighbors (KNN)", "Brecha (train-test)"]
brecha_rf  = comparacion.loc["Random Forest", "Brecha (train-test)"]

# ¿Qué clase tuvo más errores? (sobre el mejor o ambos modelos)
cm_knn = confusion_matrix(y_test, y_pred_knn)
cm_rf  = confusion_matrix(y_test, y_pred_rf)
err_por_clase_knn = {nombres_clases[i]: int(cm_knn[i].sum()-cm_knn[i,i]) for i in range(2)}
err_por_clase_rf  = {nombres_clases[i]: int(cm_rf[i].sum()-cm_rf[i,i]) for i in range(2)}

print("Mejor modelo por accuracy:", mejor_modelo)
print(f"Brecha train-test KNN: {brecha_knn:.4f} | RF: {brecha_rf:.4f}")
print("Errores por clase (KNN):", err_por_clase_knn)
print("Errores por clase (RF): ", err_por_clase_rf)


### Paso 8.2 — Respuestas a las preguntas clave

**1. ¿Cuál modelo obtuvo mejor desempeño?**  
El modelo con mejor desempeño fue **Random Forest**, con un accuracy de prueba de **0.9060**.  
KNN obtuvo **0.8980**, por lo que Random Forest fue ligeramente mejor.

**2. ¿Cuál generaliza mejor?**  
Random Forest generaliza un poco mejor porque tiene menor diferencia entre accuracy de entrenamiento y accuracy de prueba.  
La brecha de Random Forest fue **0.0940**, mientras que la de KNN fue **0.1020**.

**3. ¿Hubo overfitting?**  
Sí hay señales de overfitting en ambos modelos, porque los dos obtuvieron **1.0000** en entrenamiento y cerca de **0.90** en prueba.  
Aun así, el rendimiento en prueba sigue siendo bueno, por lo que no se considera un problema grave para esta comparación.

**4. ¿Qué ventajas tiene cada algoritmo?**  
KNN es fácil de entender y funciona bien cuando las clases están separadas por distancia. Su desventaja es que depende mucho del escalado y del valor de `k`.  
Random Forest es más robusto, maneja mejor relaciones no lineales y permite revisar la importancia de las variables.

**5. ¿Por qué accuracy no siempre es suficiente?**  
Porque si el dataset está desbalanceado, un modelo podría acertar muchas veces prediciendo casi siempre la clase mayoritaria.  
Por eso también se revisan precision, recall, F1-score y matriz de confusión.

**6. ¿Qué clase tuvo más errores?**  
En KNN, la clase con más errores fue **Ürgüp Sivrisi**, con **30** errores.  
En Random Forest los errores fueron más balanceados: **24** errores en `Çerçevelik` y **23** en `Ürgüp Sivrisi`.

**7. ¿Qué dicen precision y recall?**  
En ambos modelos, precision y recall están cerca de 0.90, lo que indica un desempeño equilibrado.  
Random Forest obtuvo valores ligeramente mejores, por eso se considera el modelo más fuerte en esta corrida.


## Paso 9 — Conclusiones

Después de realizar el análisis y entrenar los modelos, se concluye lo siguiente:

1. El dataset tiene 2500 registros, 12 variables predictoras y una variable objetivo.
2. Las clases están bastante balanceadas, por lo que accuracy puede usarse, pero no debe ser la única métrica.
3. Se encontraron pocos valores nulos y fueron reemplazados usando la mediana.
4. El escalado fue necesario, especialmente para KNN, porque este algoritmo depende de distancias.
5. KNN alcanzó un buen resultado, pero Random Forest tuvo un rendimiento ligeramente superior.
6. Random Forest obtuvo mejor accuracy, precision, recall y F1-score en prueba.
7. Ambos modelos muestran una brecha entre entrenamiento y prueba, pero Random Forest generalizó un poco mejor.
8. La variable con mayor importancia en Random Forest fue `Roundness`, seguida por `Aspect_Ration` y `Compactness`.

En general, **Random Forest fue el mejor modelo para este dataset**, aunque la diferencia con KNN no fue muy grande.
